<a href="https://colab.research.google.com/github/sakthiprasanth16/fish-image-classification/blob/main/Deployment_fish_classification_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!mv cloudflared /usr/local/bin/

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
%%writefile app.py
import os
import numpy as np
import streamlit as st
from PIL import Image
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre

# CONFIG
MODEL_PATH = "/content/drive/MyDrive/fish_models/EfficientNetB0_head_finetuned.keras"
IMG_SIZE = (256, 256)

CLASS_NAMES = [
    "animal fish",
    "animal fish bass",
    "fish sea_food black_sea_sprat",
    "fish sea_food gilt_head_bream",
    "fish sea_food hourse_mackerel",
    "fish sea_food red_mullet",
    "fish sea_food red_sea_bream",
    "fish sea_food sea_bass",
    "fish sea_food shrimp",
    "fish sea_food striped_red_mullet",
    "fish sea_food trout"
]

st.set_page_config(page_title="EfficientNet Fish Classifier", layout="centered")
st.title("🐟 Fish Classifier")

# LOAD MODEL
@st.cache_resource
def load_eff_model():
    model = load_model(MODEL_PATH, compile=False)
    return model

try:
    model = load_eff_model()
    st.success("Model loaded successfully!")
except Exception as e:
    st.error(f"Model loading failed:\n{e}")
    st.stop()


# PREPROCESS IMAGE
def prepare_image(pil_img):
    pil_img = pil_img.convert("RGB")
    pil_img = pil_img.resize((IMG_SIZE[1], IMG_SIZE[0]))
    arr = img_to_array(pil_img)        # (H, W, 3)
    arr = np.expand_dims(arr, axis=0)  # (1, H, W, 3)
    arr = eff_pre(arr)
    return arr

# PREDICT
def predict(img):
    x = prepare_image(img)
    preds = model.predict(x)[0]
    idxs = preds.argsort()[::-1][:TOP_K]
    return [(CLASS_NAMES[i], float(preds[i])) for i in idxs]


# UI
uploaded = st.file_uploader("Upload fish image", type=["jpg", "jpeg", "png"])

if uploaded:
    img = Image.open(uploaded)
    st.image(img, caption="Uploaded Image", use_container_width=True)

    if st.button("Predict"):
        results = predict(img)
        top_label, top_prob = results[0]

        st.markdown(f"## 🎯 Prediction: **{top_label}**")
        st.markdown(f"### Confidence: **{top_prob*100:.2f}%**")

Writing app.py


In [5]:
!streamlit run app.py &>/dev/null &

In [ ]:
!cloudflared tunnel --url http://localhost:8501

In [ ]:
!pkill streamlit
!pkill cloudflared